# Ü-I — Iceberg-Tabelle in Glue erzeugen & im Catalog anlegen (LÖSUNG)

Aus der katalogisierten Rohtabelle `raw.orders` eine **Apache-Iceberg**-Tabelle
`processed.orders_iceberg` erzeugen und im **Glue Data Catalog** registrieren —
danach direkt aus Athena abfragbar.

**Kern:** Format + Catalog-Anbindung (`USING iceberg` + `GlueCatalog`-Impl), nicht
der ETL-Inhalt. Iceberg ist in Glue 5.0 nativ, aktiviert über `--datalake-formats=iceberg`.

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
# --datalake-formats=iceberg lädt die Iceberg-Jars; die --conf-Kette verdrahtet
# den named Catalog `glue_catalog` mit dem Glue Data Catalog. Muss beim
# Session-Bau stehen (spark.sql.extensions lässt sich später nicht mehr setzen).
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://gfu-glue-training-629452195361/processed/ --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO"
}

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Iceberg-Tabelle anlegen — und damit im Catalog registrieren

`USING iceberg` macht es zur Iceberg-Tabelle; `CREATE TABLE` über den
`glue_catalog` schreibt den Eintrag sofort in den Data Catalog (kein Crawler).
`LOCATION` legt die Dateien in `processed/orders_iceberg/` ab.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS glue_catalog.processed.orders_iceberg (
        customer_id string,
        order_total double,
        order_date  string,
        status      string
    )
    USING iceberg
    LOCATION 's3://gfu-glue-training-629452195361/processed/orders_iceberg'
    TBLPROPERTIES ('format-version' = '2')
""")

## 2) Aus `raw.orders` befüllen

Rohspalten mit Leerzeichen → Backticks. `CAST(\`order total\` AS double)` macht
die leere Zelle (C004, C005) zu NULL. `raw.orders` liegt im Default-Catalog.

In [ ]:
spark.sql("""
    INSERT INTO glue_catalog.processed.orders_iceberg
    SELECT
        `cust id`                     AS customer_id,
        CAST(`order total` AS double) AS order_total,
        `order date`                  AS order_date,
        status                        AS status
    FROM raw.orders
""")

## 3) Prüfen — Row-Count, Iceberg-Provider, Snapshot

In [ ]:
print("rows:", spark.sql("SELECT count(*) AS n FROM glue_catalog.processed.orders_iceberg").collect()[0]["n"])  # 48
spark.sql("DESCRIBE EXTENDED glue_catalog.processed.orders_iceberg").show(100, truncate=False)  # Provider: iceberg
spark.sql("SELECT snapshot_id, operation, summary FROM glue_catalog.processed.orders_iceberg.snapshots").show(truncate=False)

## 4) Gegencheck in Athena (als Trainee)

Athena v3 liest Iceberg über den Catalog nativ — kein Extra-Setup:

```sql
SELECT status, count(*) FROM processed.orders_iceberg GROUP BY status;
```

**Aufräumen:** `DROP TABLE glue_catalog.processed.orders_iceberg PURGE;`